## Example 1: the package and its entry points

In [ ]:
import dbgpu
from dbgpu import GPUDatabase, GPUSpecification

# Example 1: dbgpu is a database of GPU specifications (names, architectures, compute capability,
# memory, clocks). It does not run anything on a GPU. Its entry points are GPUDatabase and GPUSpecification.
print("Example 1: the package and its entry points")
for name in ("GPUDatabase", "GPUSpecification", "ALL_MANUFACTURERS", "DEFAULT_GPU_DB_PATH"):
    assert hasattr(dbgpu, name), f"dbgpu has no {name}"
assert isinstance(GPUDatabase, type) and isinstance(GPUSpecification, type)
assert callable(GPUDatabase.default) and callable(GPUDatabase.from_file)
assert "NVIDIA" in dbgpu.ALL_MANUFACTURERS, dbgpu.ALL_MANUFACTURERS
print(f"  dbgpu {dbgpu.__version__}")
print(f"  manufacturers covered: {list(dbgpu.ALL_MANUFACTURERS)}")
print("Example 1 completed\n")

## Example 2: Advanced Operations 2

In [ ]:
import os
import dbgpu
from dbgpu import GPUDatabase, GPUSpecification

# Example 2: dbgpu is a GPU-SPECIFICATION database built from TechPowerUp. The data ships inside the
# package (data.pkl), so GPUDatabase.default() works offline; it is loaded once, then cached.
print("Example 2: Load the bundled GPU database and look up GPUs by name")
assert os.path.isfile(dbgpu.DEFAULT_GPU_DB_PATH), dbgpu.DEFAULT_GPU_DB_PATH
db = GPUDatabase.default()
assert GPUDatabase.default() is db
assert len(db.specs) >= 2800, f"only {len(db.specs)} specs in the bundled database"  # measured 2824 in 2025.12
assert all(isinstance(s, GPUSpecification) for s in db.specs)
print(f"  {len(db.specs)} GPU specifications in {os.path.basename(dbgpu.DEFAULT_GPU_DB_PATH)}")

# A lookup normalises the name: bare, manufacturer-prefixed and safe-key spellings resolve to ONE spec
gb10 = db["GB10"]
assert db["NVIDIA GB10"] is gb10 and db["gb10"] is gb10
assert (gb10.manufacturer, gb10.gpu_name, gb10.architecture) == ("NVIDIA", "GB20B", "Blackwell 2.0")
assert (gb10.cuda_major_version, gb10.cuda_minor_version) == (12, 1)  # compute capability 12.1, the DGX Spark GPU
assert (gb10.streaming_multiprocessors, gb10.memory_size_gb, gb10.memory_type) == (48, 128.0, "LPDDR5X")
print(f"  {gb10.name}: {gb10.architecture}, CUDA {gb10.cuda_major_version}.{gb10.cuda_minor_version}, "
      f"{gb10.streaming_multiprocessors} SMs, {gb10.memory_size_gb:.0f} GB {gb10.memory_type}")

# An unknown name raises KeyError carrying the normalised key
try:
    db["Not A Real GPU 9000"]
except KeyError as e:
    assert e.args == ("not-a-real-gpu-9000",), e.args
    print(f"  unknown name rejected as expected: KeyError({e.args[0]!r})")
else:
    raise AssertionError("GPUDatabase returned a spec for a GPU name it does not contain")
print("Example 2 completed\n")

## Example 3: Advanced Operations 3

In [ ]:
from dbgpu import GPUDatabase

# Example 3: query the database — filter the specifications by any field (plain comprehensions over db.specs)
print("Example 3: Filter GPU specifications by architecture, compute capability and memory")
db = GPUDatabase.default()
specs = db.specs

hopper = sorted(s.name for s in specs if s.architecture == "Hopper")
assert {"H100 SXM5 80 GB", "H100 PCIe 80 GB", "H200 SXM 141 GB"} <= set(hopper), hopper  # measured 12 Hopper parts
assert all((db[n].manufacturer, db[n].cuda_major_version) == ("NVIDIA", 9) for n in hopper), hopper
print(f"  Hopper: {len(hopper)} parts, e.g. {hopper[:3]}")

def compute_capability(spec):
    return (spec.cuda_major_version, spec.cuda_minor_version)

cc_121 = [s.name for s in specs if compute_capability(s) == (12, 1)]
assert "GB10" in cc_121, cc_121  # measured in 2025.12: the only 12.1 entry
turing_75 = [s for s in specs if compute_capability(s) == (7, 5)]
assert "Tesla T4G" in [s.name for s in turing_75], f"Tesla T4G missing from the {len(turing_75)} compute-7.5 parts"  # measured 86
assert all(s.architecture == "Turing" for s in turing_75), {s.architecture for s in turing_75}
print(f"  compute capability 12.1: GB10; 7.5: {len(turing_75)} Turing parts including Tesla T4G")

big = sorted((s for s in specs if (s.memory_size_gb or 0) >= 128), key=lambda s: s.memory_size_gb, reverse=True)
assert "GB10" in [s.name for s in big] and big[0].memory_size_gb >= 288.0, [(s.name, s.memory_size_gb) for s in big]
print(f"  {len(big)} parts with >= 128 GB; largest: {big[0].name} ({big[0].memory_size_gb:.0f} GB)")  # measured 288 GB
print("Example 3 completed\n")

## Example 4: Advanced Operations 4

In [ ]:
import collections
import dbgpu
from dbgpu import GPUDatabase

# Example 4: analytics — the whole database as a pandas DataFrame, and a field-by-field GPU comparison
print("Example 4: Analytics with db.dataframe and labeled_comparison_fields")
db = GPUDatabase.default()
df = db.dataframe
assert len(df) == len(db.specs) and {"manufacturer", "architecture", "memory_size_gb"} <= set(df.columns)
per_vendor = df.groupby("manufacturer").size().to_dict()
assert per_vendor == dict(collections.Counter(s.manufacturer for s in db.specs)), per_vendor
assert set(per_vendor) <= set(dbgpu.ALL_MANUFACTURERS) and per_vendor["NVIDIA"] >= 1200  # measured 1286
print(f"  parts per manufacturer: {per_vendor}")

# What `dbgpu compare` prints: the first GPU's labelled fields, and the second GPU's delta against it
rtx4090, rtx3090 = db["GeForce RTX 4090"], db["GeForce RTX 3090"]
fields, delta = dict(rtx4090.labeled_fields()), dict(rtx4090.labeled_comparison_fields(rtx3090))
assert (fields["Architecture"], fields["Memory Size"], fields["Memory Bus"]) == ("Ada Lovelace", "24.0 GB", "384 bit"), fields
assert (delta["Architecture"], delta["Memory Size"], delta["Memory Bus"]) == ("!=", "", ""), delta  # both 24 GB on 384 bit
assert rtx3090.shading_units == 10496 and delta["Shading Units"] == "-5888 (-35.9%)", delta["Shading Units"]
print(f"  RTX 3090 vs RTX 4090: architecture {delta['Architecture']}, shading units {delta['Shading Units']}")
print("Example 4 completed\n")

## Example 5: Advanced Operations 5

In [ ]:
import json
import os
import subprocess
import sys
import tempfile
import dbgpu
from dbgpu import GPUDatabase

# Example 5: save and reload the database, and drive the `dbgpu` command-line tool on the bundled data
print("Example 5: JSON round trip with GPUDatabase.from_file, and the dbgpu CLI")
db = GPUDatabase.default()
with tempfile.TemporaryDirectory() as tmp:
    path = os.path.join(tmp, "gpus.json")
    with open(path, "w") as f:
        json.dump([s.to_dict() for s in db.specs], f, default=dbgpu.json_serialize)  # dates -> ISO strings
    loaded = GPUDatabase.from_file(path)
# from_file reads "" as None (measured: 1309 blank display_connectors); every other field survives, dates re-parsed
expected = {k: {f: (None if v == "" else v) for f, v in s.to_dict().items()} for k, s in db.specifications.items()}
assert {k: s.to_dict() for k, s in loaded.specifications.items()} == expected
assert loaded["GB10"] == db["GB10"]
print(f"  {len(loaded.specs)} specs reloaded from JSON; GB10 release date {loaded['GB10'].release_date}")

cli = [sys.executable, "-m", "dbgpu"]
version = subprocess.run(cli + ["--version"], capture_output=True, text=True, check=True).stdout.strip()
assert version == dbgpu.__version__, (version, dbgpu.__version__)
lookup = subprocess.run(cli + ["lookup", "Tesla T4G"], capture_output=True, text=True, check=True).stdout
assert lookup == str(db["Tesla T4G"]) + "\n" and "TU104" in lookup, lookup  # the CLI echoes str(spec)
print(f"  dbgpu {version}: `dbgpu lookup 'Tesla T4G'` printed the spec table ({len(lookup.splitlines())} lines)")
print("Example 5 completed")